# 00 · Configuración del Lakehouse y capa Bronze

**Proyecto:** Minería aurífera y deforestación en el corredor La Pampa, Huepetuhe y Mazuko (Madre de Dios, Perú), 2018-2024.

Este notebook prepara la infraestructura en Databricks y registra los datos crudos:

1. Crea el esquema y el Volume en Unity Catalog.
2. Inventaría las teselas Sentinel-2 y Hansen subidas con la Databricks CLI.
3. Registra metadatos y capas vectoriales como tablas Delta (capa **Bronze**).
4. Muestra una vista rápida de una escena para validar la ingesta.

**Arquitectura medallion**

| Capa | Contenido | Granularidad |
|---|---|---|
| Bronze | Teselas GeoTIFF crudas + metadatos STAC + vectores | archivo |
| Silver | Compuestos anuales sin nubes e índices espectrales | píxel de 20 m |
| Gold | Indicadores por hexágono H3 + covariables | hexágono (~0.1 km²) |

In [ ]:
%pip install -q rasterio pyproj h3 geopandas shapely esda libpysal spreg pykrige scikit-gstat mapclassify
dbutils.library.restartPython()

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Raíz del repositorio (Git folder en Databricks o clon local)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from geocloud import config as C, dbx
spark = dbx.get_spark()
VOL = dbx.volume_path()
FIG = Path(VOL) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Databricks:", dbx.is_databricks(), "| Volume:", VOL)

## 1. Esquema y Volume en Unity Catalog

In [ ]:
dbx.setup_catalog(spark)
grid = C.Grid()
print(f"Grilla: {grid.width} x {grid.height} píxeles a {grid.res} m = {grid.width*grid.height/1e6:.1f} M píxeles por banda")
print(f"Teselas de procesamiento: {grid.n_tiles[0]} x {grid.n_tiles[1]} de {grid.tile} px")

## 2. Inventario de archivos (sin leer su contenido)

`binaryFile` lista los archivos del Volume como un DataFrame distribuido. Spark infiere
las columnas `year` y `tile` a partir de la estructura de carpetas `year=YYYY/tile=rRR_cCC`
(particionamiento estilo Hive), lo que permite filtrar sin abrir los archivos.

In [ ]:
from pyspark.sql import functions as F

s2_files = (spark.read.format("binaryFile")
            .option("pathGlobFilter", "*.tif")
            .load(f"{VOL}/bronze/s2")
            .drop("content")
            .withColumn("item_id", F.regexp_extract("path", r"([^/]+)\.tif$", 1)))

resumen = (s2_files.groupBy("year")
           .agg(F.countDistinct("item_id").alias("escenas"),
                F.count("*").alias("teselas"),
                F.round(F.sum("length") / 1e9, 2).alias("GB"))
           .orderBy("year"))
dbx.show(resumen)

In [ ]:
tot = s2_files.agg(F.count("*").alias("n"), F.sum("length").alias("b")).first()
n_px_crudos = s2_files.groupBy("year", "item_id").count().count() * grid.width * grid.height * 7
print(f"Archivos: {tot.n:,} | Volumen comprimido: {tot.b/1e9:.2f} GB")
print(f"Valores raster crudos a procesar (escenas x píxeles x bandas): {n_px_crudos/1e9:.2f} mil millones")

## 3. Tablas Bronze: metadatos de escenas, inventario y vectores

In [ ]:
import pandas as pd

items = pd.read_csv(f"{VOL}/bronze/s2_items.csv")
items = items[items["status"] == "descargado"]
bronze_items = spark.createDataFrame(items.astype({"year": int}))
dbx.save_table(bronze_items, "bronze_s2_items")
dbx.save_table(s2_files, "bronze_s2_files")
dbx.show(spark.table(dbx.tbl("bronze_s2_items")).orderBy("year", "cloud_cover").limit(10))

In [ ]:
import geopandas as gpd

vec_dir = Path(VOL) / "bronze" / "vector"
for f in sorted(vec_dir.glob("*.geojson")):
    g = gpd.read_file(f)
    g["wkt"] = g.geometry.to_wkt()
    pdf = pd.DataFrame(g.drop(columns="geometry")).astype(str)
    name = "bronze_vec_" + f.stem
    dbx.save_table(spark.createDataFrame(pdf), name)
    print(f"{name:35s} {len(g):5d} geometrías")

## 4. Control de calidad visual: color verdadero de una tesela

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio

sample = (spark.table(dbx.tbl("bronze_s2_files"))
          .orderBy(F.desc("length")).select("path").first().path.replace("dbfs:", "").replace("file:", ""))
with rasterio.open(sample) as ds:
    arr = ds.read().astype(float)
rgb = np.dstack([arr[2], arr[1], arr[0]]) / 10000
rgb = np.clip(rgb / np.nanpercentile(rgb[rgb > 0], 98), 0, 1)
scl = arr[6]

fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].imshow(rgb); ax[0].set_title("Color verdadero (B4, B3, B2)")
im = ax[1].imshow(scl, cmap="tab20", vmin=0, vmax=11); ax[1].set_title("Scene Classification Layer (SCL)")
plt.colorbar(im, ax=ax[1], fraction=0.046)
for a in ax: a.axis("off")
plt.suptitle(Path(sample).name); plt.tight_layout()
plt.savefig(FIG / "00_quicklook.png", dpi=150); plt.show()

**Lectura de SCL:** 4 = vegetación, 5 = suelo desnudo, 6 = agua, 8-10 = nubes y cirros, 3 = sombra de nube.
Solo las clases 4-7 entran al compuesto; el resto se enmascara píxel a píxel en la capa Silver.